> ☁️ **This notebook runs against the free shared sandbox** — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")`. It uses features that need a server (fleet delegation, category allocations, shadow mode, or cross-process budgets). The sandbox is for evaluation only — **not for production** (may be wiped at any time; no SLA).
>
> **Three ways to run FiGuard — same API, same rules. Swap only the client constructor:**
>
> - ☁️ **Free sandbox** _(this notebook)_ — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")` — quick hosted evaluation.
> - 🏢 **Self-hosted server** — `FiGuardClient(base_url="https://your-figuard", api_key="...")` — your data, your infra; the production path for the server features this notebook uses. [Self-host →](https://github.com/figuard/figuard-core#self-hosting)
> - 💻 **Embedded** — `FiGuardClient()` — in-process, zero infra (single process; the server-only features used here aren't available embedded).

# Per-client cost attribution for an agency

**The scenario.** An agency runs AI agents for its clients and needs to show each client *what their agents did, what it cost, and where a runaway was caught* — broken down by category.

FiGuard gives you that as a by-product of enforcement: one budget per client, a hard cap per category, and an append-only ledger that records every authorized, confirmed, and **denied** call with its `category`. This notebook builds a per-category client report straight from that ledger — no extra instrumentation.

The denial is the interesting line: a $0 event that proves you caught a loop *before* it cost the client anything.

In [ ]:
# @title Step 1 — Install and configure FiGuard
import sys, importlib
!pip install "figuard>=1.1.2" --upgrade -q
# Flush any stale cached module from this runtime session
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")
import uuid
CLIENT_ID = f"client_acme_{uuid.uuid4().hex[:6]}"
print(f"Client workspace: {CLIENT_ID}  (identifies this client's budget in the dashboard)")

## Step 2 — One budget per client, a hard cap per category

The agency gives this client a $500 ceiling, split into category caps via delegation tokens — one per sub-agent. Each agent can only spend within its category, and a runaway in one category can't drain another.

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(base_url=FIGUARD_BASE_URL, api_key="sb_live_demo")  # zero-config: connects to the public sandbox

# The client's monthly budget
budget = client.create_budget(
    user_id=CLIENT_ID,
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    intent_context="Acme Corp — agents run by the agency",
)
print(f"✓ Client budget: {budget.id}  (${budget.total_limit:.0f} ceiling)")

# A delegation token per agent, each capped to its category
research_token = client.create_delegation_token(
    budget_id=budget.id, label="research_agent",
    caps=[{"category": "research", "limit": 200.00}],
)
outreach_token = client.create_delegation_token(
    budget_id=budget.id, label="outreach_agent",
    caps=[{"category": "outreach", "limit": 300.00}],
)
print("✓ research_agent  cap $200 (research)")
print("✓ outreach_agent  cap $300 (outreach)")

## Optional — wire it live to AgentWatch (real webhooks)

Everything above runs on the sandbox and builds the report from the **ledger**. To instead push each spend event into a downstream system like AgentWatch **in real time**, register a webhook — and run this against **your own** FiGuard (`docker compose up`), not the shared sandbox:

```bash
export FIGUARD_BASE_URL=http://localhost:8080
export AGENTWATCH_URL=https://agentwatch-api.up.railway.app
```

The next cell registers the webhook **before** the agents run, so every confirm/denial below is delivered live. FiGuard splits denials across event types, so it subscribes to **all** of them — `SPEND_DENIED` (cap/funds), `VELOCITY_LIMIT_EXCEEDED` (velocity loop), `ANOMALY_DETECTED` (anomaly) — otherwise a velocity-based loop-kill is silently dropped.

In [ ]:
# OPTIONAL — wire each spend event to AgentWatch live. Runs only against YOUR OWN
# FiGuard (FIGUARD_BASE_URL set), never the shared sandbox.
import os
AGENTWATCH_URL = os.environ.get("AGENTWATCH_URL")
on_sandbox = not os.environ.get("FIGUARD_BASE_URL")
webhook = None
# FiGuard splits "agent stopped" across event types — subscribe to ALL so you never miss a
# velocity loop-kill or anomaly auto-pause (neither is a SPEND_DENIED):
STOP_EVENTS = ["SPEND_CONFIRMED", "SPEND_DENIED", "VELOCITY_LIMIT_EXCEEDED", "ANOMALY_DETECTED"]
if AGENTWATCH_URL and not on_sandbox:
    hook_url = AGENTWATCH_URL.rstrip("/") + "/webhooks/figuard"
    try:  # avoid duplicate webhooks across re-runs
        for w in client.list_webhooks():
            if getattr(w, "url", "") == hook_url:
                client.delete_webhook(w.id)
    except Exception:
        pass
    webhook = client.create_webhook(url=hook_url, secret="figuard-agentwatch-key-0001",
                                    events=STOP_EVENTS)
    print(f"✓ live wiring on — every spend event below fires to {hook_url}")
elif AGENTWATCH_URL and on_sandbox:
    print("⚠ AGENTWATCH_URL set but you're on the SHARED sandbox — skipping (would leak other "
          "users' events). Run your own FiGuard: export FIGUARD_BASE_URL=http://localhost:8080")
else:
    print("(ledger-only — set AGENTWATCH_URL + FIGUARD_BASE_URL=local to wire AgentWatch live)")

## Step 3 — Run the agents (and catch a loop)

The outreach agent does its job. The research agent gets stuck re-querying — "one more source" — and tries to blow past its $200 cap. FiGuard denies the overrun. The client is never billed for it.

In [ ]:
def run(token, agent, category, amount, desc, key):
    a = client.authorize(
        session_token=token, agent_id=agent, action_type="EXTERNAL_CALL",
        description=desc, requested_quantity=amount,
        claimed_category=category, idempotency_key=key,
    )
    if a.is_authorized:
        client.confirm_event(a.event_id, confirmed_quantity=amount)
        print(f"  [OK]  {agent:<15} {category:<9} ${amount:>6.2f}  {desc}")
    else:
        print(f"  [X]   {agent:<15} {category:<9} ${amount:>6.2f}  DENIED — {a.denial_reason}")
    return a

print("outreach_agent")
run(outreach_token.session_token, "outreach_agent", "outreach", 100.00, "Email sequence — batch 1", "out-1")
run(outreach_token.session_token, "outreach_agent", "outreach", 100.00, "Email sequence — batch 2", "out-2")

print("\nresearch_agent")
run(research_token.session_token, "research_agent", "research", 50.00, "Market scan — query 1", "res-1")
run(research_token.session_token, "research_agent", "research", 50.00, "Market scan — query 2", "res-2")
run(research_token.session_token, "research_agent", "research", 50.00, "Market scan — query 3", "res-3")
# loop: agent decides it needs 'one more source' — would push research to $250, over the $200 cap
run(research_token.session_token, "research_agent", "research", 100.00, "Loop: 'one more source'", "res-4")

## Step 4 — The client report, built from the ledger

Every call above is in FiGuard's append-only ledger, tagged with its `category`. Group by category and you have the report: confirmed spend, attempts, and the denial that proves the loop was caught — 100% of events, not sampled.

In [ ]:
from collections import defaultdict

page = client.get_ledger(budget.id, page=0, size=50)

report = defaultdict(lambda: {"confirmed": 0.0, "calls": 0, "denied": 0})
for ev in page.events:
    cat = ev.claimed_category or "(uncategorized)"
    if ev.decision == "CONFIRMED":
        report[cat]["confirmed"] += float(ev.confirmed_quantity or 0)
        report[cat]["calls"] += 1
    elif ev.decision == "DENIED":
        report[cat]["denied"] += 1

print(f"Client report — {CLIENT_ID}")
print(f"{'CATEGORY':<12}{'SPENT':>10}{'CALLS':>8}{'BLOCKED':>9}")
print("-" * 39)
total = 0.0
for cat, r in sorted(report.items()):
    print(f"{cat:<12}{'$'+format(r['confirmed'],'.2f'):>10}{r['calls']:>8}{r['denied']:>9}")
    total += r["confirmed"]
print("-" * 39)
print(f"{'TOTAL':<12}{'$'+format(total,'.2f'):>10}")
print(f"\nOf a ${budget.total_limit:.0f} ceiling. The blocked call is the loop FiGuard caught — $0 to the client.")
print(f"\n→ Spend tree: https://figuard-sandbox-g1ha.onrender.com/ui/budgets/{budget.id}")

## Optional — forward to your reporting / observability stack

The table above is the *data*. To turn it into a live, client-facing report, point a FiGuard webhook at your reporting system — FiGuard fires `SPEND_CONFIRMED` plus the denial events (`SPEND_DENIED`, `VELOCITY_LIMIT_EXCEEDED`, `ANOMALY_DETECTED`) on every event, and **both payloads carry `category`** (plus `agentId`, amounts, and the denial reason), so your system maps straight from them. No polling, no new code on the FiGuard side.

```python
# Register once per tenant; FiGuard POSTs signed events to your URL.
client.create_webhook(
    url="https://your-reporting-system.example.com/webhooks/figuard",
    secret="a-signing-secret-min-16-chars",  # verify deliveries with verify_webhook()
    events=["SPEND_CONFIRMED", "SPEND_DENIED", "VELOCITY_LIMIT_EXCEEDED", "ANOMALY_DETECTED"],
)
```

### Payloads your handler receives

`SPEND_CONFIRMED`:

```json
{
  "eventType": "SPEND_CONFIRMED",
  "budgetId": "0e004964-...", "tenantId": "...", "userId": "client_acme",
  "currency": "USD", "unit": null, "timestamp": "2026-06-11T00:45:28Z",
  "spendEventId": "2538dc3c-...",
  "requestedQuantity": 50.00,
  "confirmedQuantity": 50.00,
  "agentId": "research_agent",
  "category": "research",
  "totalLimit": 500.00,
  "quantitySpent": 150.00,
  "availableQuantity": 350.00
}
```

`SPEND_DENIED` (the $0 event that proves a loop was caught):

```json
{
  "eventType": "SPEND_DENIED",
  "budgetId": "0e004964-...", "tenantId": "...", "userId": "client_acme",
  "currency": "USD", "unit": null, "timestamp": "2026-06-11T00:45:41Z",
  "spendEventId": "91eabe59-...",
  "requestedQuantity": 100.00,
  "denialReason": "DELEGATE_CAP_EXCEEDED",
  "denialMessage": "Delegation token cap for 'research' has 50.0000 available, requested 100",
  "agentId": "research_agent",
  "category": "research"
}
```

### Mapping into a report

| Report field        | From the payload                                         |
|---------------------|----------------------------------------------------------|
| decision            | `eventType` — `SPEND_CONFIRMED` -> confirmed, `SPEND_DENIED` -> denied |
| cost                | `confirmedQuantity` (denials are $0)                     |
| client / workspace  | `userId`                                                 |
| agent attribution   | `agentId`                                                |
| category breakdown  | `category`                                               |
| "caught a loop"     | a `SPEND_DENIED` row — $0 cost, with `denialReason`      |

FiGuard enforces the budget *before* the cost lands; your reporting layer explains what happened *after*. Same data, two audiences — the agency's controls and the client's report.

In [ ]:
# If wired live, confirm delivery + point at AgentWatch's report
if webhook is not None:
    import time; time.sleep(5)
    print("FiGuard → AgentWatch deliveries (your endpoint's HTTP responses):")
    for d in client.get_webhook_deliveries(webhook.id):
        print(f"  {d.event_type:<22} HTTP {d.response_status}")
    print(f"\nAgentWatch report:  {AGENTWATCH_URL.rstrip('/')}/report/{budget.id}")
    print(f"  session_id = {budget.id}   workspace_id = {CLIENT_ID}")